# SIMILIS baseline: fields, vocabularies, template

Выбор 2–4 полей для baseline, нормализация словарей и проектирование шаблона `auto_description`

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
from IPython.display import display

In [2]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
MANIFEST_PATH = PROJECT_ROOT / 'data' / 'interim' / 'manifest_raw.csv'
REPORTS_DIR = PROJECT_ROOT / 'artifacts' / 'reports'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(MANIFEST_PATH)
df = df[df['image_exists'] == True].copy()
df['description'] = df['description'].fillna('')

print('Rows for analysis:', len(df))

Rows for analysis: 1387


## Task 5. Structure of descriptions and baseline field choice

In [3]:
sample_desc = df[['code', 'name', 'material', 'fragm', 'description']].sample(20, random_state=42)
display(sample_desc)

,code,name,material,fragm,description
651,Нц-34-079,Изразец,Керамика,Фрагмент,Изразца стенового с белой поливой фрагмент
208,БП12-Г01-7212,Изразец,Керамика,Фрагмент,Изразца красноглиняного декорированного фр-т
949,Нейш3А-2018-0833,Блюдце,Фарфор,Фрагмент,Блюдца фарфорового с цветочным орнаментом внут...
680,Нц-37-0589,Изразец,Керамика,Фрагмент,Изразца красноглиняного с белой поливой фр-т
805,Нц-45-1466,Изразец,Керамика,Фрагмент,Изразца красноглиняного со следами темной поли...
239,БП12-Г02-4190,Тарелка,Фаянс,Фрагмент,Тарелки фаянсовой с кобальтовой росписью и рел...
866,Нц-48-376,Изразец,Керамика,Фрагмент,"Изразца печного красноглиняного ""муравленого"" ..."
529,Нц-23-2188,Изразец,Глина,Фрагмент,Изразца шведского красноглиняного с растительн...
723,Нц-45-0018,Тарелка,Фаянс,Фрагмент,Тарелки фаянсовой с белой поливой профиля фр-т
766,Нц-45-0558,Тарелка,Фаянс,Фрагмент,Тарелки фаянсовой венчика фр-т


In [4]:
field_review = pd.DataFrame([
    {
        'field_name': 'object_type',
        'why_visually_observable': 'Общая форма и класс предмета обычно различимы по силуэту и макроморфологии.',
        'target_type': 'single-label',
        'rough_num_classes': int(df['name'].nunique()),
        'missing_share': float(df['name'].isna().mean()),
        'ambiguity_risk': 'medium',
    },
    {
        'field_name': 'part_zone',
        'why_visually_observable': 'Для фрагментов часто видны венчик, донце, профиль, крышка и другие зоны.',
        'target_type': 'single-label',
        'rough_num_classes': 6,
        'missing_share': float('nan'),
        'ambiguity_risk': 'medium-high',
    },
    {
        'field_name': 'integrity',
        'why_visually_observable': 'Целый предмет или фрагмент обычно определяется напрямую по изображению.',
        'target_type': 'single-label',
        'rough_num_classes': int(df['fragm'].nunique()),
        'missing_share': float(df['fragm'].isna().mean()),
        'ambiguity_risk': 'low',
    },
    {
        'field_name': 'material_group',
        'why_visually_observable': 'Материал часто извлекается вероятностно по фактуре, цвету и блеску, но требует грубого словаря.',
        'target_type': 'single-label',
        'rough_num_classes': 6,
        'missing_share': float(df['material'].isna().mean()),
        'ambiguity_risk': 'medium',
    },
    {
        'field_name': 'function',
        'why_visually_observable': 'Часто зависит не только от изображения, но и от внешнего контекста.',
        'target_type': 'better to exclude',
        'rough_num_classes': np.nan,
        'missing_share': np.nan,
        'ambiguity_risk': 'high',
    },
    {
        'field_name': 'dating_or_culture',
        'why_visually_observable': 'Не извлекается надёжно по одному изображению без экспертного контекста.',
        'target_type': 'better to exclude',
        'rough_num_classes': np.nan,
        'missing_share': np.nan,
        'ambiguity_risk': 'high',
    },
])
display(field_review)

,field_name,why_visually_observable,target_type,rough_num_classes,missing_share,ambiguity_risk
0,object_type,Общая форма и класс предмета обычно различимы ...,single-label,10.0,0.0,medium
1,part_zone,"Для фрагментов часто видны венчик, донце, проф...",single-label,6.0,NaN,medium-high
2,integrity,Целый предмет или фрагмент обычно определяется...,single-label,2.0,0.0,low
3,material_group,Материал часто извлекается вероятностно по фак...,single-label,6.0,0.0,medium
4,function,"Часто зависит не только от изображения, но и о...",better to exclude,NaN,NaN,high
5,dating_or_culture,Не извлекается надёжно по одному изображению б...,better to exclude,NaN,NaN,high


Выбранные поля для baseline:

- `object_type` из `name`
- `integrity` из `fragm`
- `material_group` из `material` после нормализации
- `part_zone` из `description` по ограниченному словарю

Поля, которые сознательно не берём в baseline:

- функция предмета;
- датировка;
- культурная интерпретация;
- тонкие типологические метки;
- ссылки на отчёты и внешнюю документацию.

Причина: эти поля либо не наблюдаемы напрямую по одному изображению, либо требуют гораздо более сильного экспертного контекста.

## Task 6. Vocabulary normalization and target columns

In [5]:
object_type_map = {
    'Тарелка': 'тарелка',
    'Тарелка/блюдо': 'тарелка',
    'Блюдце': 'блюдце',
    'Миска': 'миска',
    'Миска (?)': 'миска',
    'Крышка': 'крышка',
    'Изразец': 'изразец',
    'Плитка': 'плитка',
    'Игрушка': 'игрушка',
    'Игрушка ёлочная': 'игрушка',
}

material_map = {
    'Керамика': 'ceramic',
    'Красноглиняная керамика': 'ceramic',
    'Белоглиняная керамика': 'ceramic',
    'Светлоглиняная керамика': 'ceramic',
    'Серолощеная керамика': 'ceramic',
    'Глина': 'clay',
    'Фаянс': 'faience',
    'Фарфор': 'porcelain',
    'Стекло': 'glass',
    'Каменная масса': 'stoneware',
    'Камень': 'stone',
    'Песчаник': 'stone',
    'Сланец': 'stone',
    'Гранит': 'stone',
    'Дерево': 'other',
    'Пластмасса': 'other',
    'Резина': 'other',
    'Металл': 'metal',
    'Белый металл': 'metal',
    'Бронза': 'metal',
    'Основание - бетон, покрытие - композитный материал': 'other',
}

integrity_map = {
    'Фрагмент': 'fragment',
    'Целый': 'whole',
}

print('Object type normalization:')
display(pd.DataFrame(object_type_map.items(), columns=['raw_label', 'norm_label']))

print('Material normalization:')
display(pd.DataFrame(material_map.items(), columns=['raw_label', 'norm_label']).head(30))

Object type normalization:


,raw_label,norm_label
0,Тарелка,тарелка
1,Тарелка/блюдо,тарелка
2,Блюдце,блюдце
3,Миска,миска
4,Миска (?),миска
5,Крышка,крышка
6,Изразец,изразец
7,Плитка,плитка
8,Игрушка,игрушка
9,Игрушка ёлочная,игрушка


Material normalization:


,raw_label,norm_label
0,Керамика,ceramic
1,Красноглиняная керамика,ceramic
2,Белоглиняная керамика,ceramic
3,Светлоглиняная керамика,ceramic
4,Серолощеная керамика,ceramic
5,Глина,clay
6,Фаянс,faience
7,Фарфор,porcelain
8,Стекло,glass
9,Каменная масса,stoneware


In [6]:
part_zone_rules = [
    ('донце', 'base'),
    ('донца', 'base'),
    ('венчик', 'rim'),
    ('края', 'rim'),
    ('профиль', 'profile'),
    ('стенка', 'wall'),
    ('стенки', 'wall'),
    ('крышка', 'lid'),
    ('крышечки', 'lid'),
    ('крышки', 'lid'),
    ('ручка', 'handle'),
    ('ручки', 'handle'),
]

def extract_part_zone(text: str) -> str:
    text = (text or '').lower()
    for needle, label in part_zone_rules:
        if needle in text:
            return label
    return 'unknown'

df['part_zone_raw'] = df['description'].map(extract_part_zone)
df['object_type_raw'] = df['name']
df['material_raw'] = df['material']
df['integrity_raw'] = df['fragm']

df['object_type'] = df['object_type_raw'].map(object_type_map).fillna('other')
df['material_group'] = df['material_raw'].map(material_map).fillna('other')
df['integrity'] = df['integrity_raw'].map(integrity_map).fillna('unknown')
df['part_zone'] = df['part_zone_raw']

df['label_is_missing_part_zone'] = df['part_zone'].eq('unknown')
df['label_is_missing_material'] = df['material_raw'].isna()
df['label_is_uncertain_object_type'] = df['name'].astype(str).str.contains(r'\?|/', regex=True)
df['label_is_uncertain_part_zone'] = False

display(df[['code','object_type_raw','object_type','material_raw','material_group','integrity_raw','integrity','description','part_zone']].head(10))

,code,object_type_raw,object_type,material_raw,material_group,integrity_raw,integrity,description,part_zone
0,М102-2012-1-0494,Изразец,изразец,Керамика,ceramic,Фрагмент,fragment,Изразца красноглиняного с белой поливой и коба...,unknown
1,М102-2012-1-0496,Изразец,изразец,Керамика,ceramic,Фрагмент,fragment,Изразца красноглиняного плоского с белой полив...,unknown
2,М102-2012-1-0529,Изразец,изразец,Керамика,ceramic,Фрагмент,fragment,Изразца-перемычки красноглиняного с белой поли...,unknown
3,М102-2012-1-0585,Тарелка,тарелка,Фаянс,faience,Фрагмент,fragment,Сосуда (тарелки ?) фаянсового с белой поливой ...,rim
4,М102-2012-1-0603,Изразец,изразец,Керамика,ceramic,Фрагмент,fragment,Изразца (перемычки?) красноглиняного с белой п...,unknown
5,М102-2012-1-0611,Изразец,изразец,Керамика,ceramic,Фрагмент,fragment,Изразца красноглиняного плоского с белой полив...,unknown
6,М102-2012-1-0691,Миска,миска,Керамика,ceramic,Фрагмент,fragment,"Миски серолощеной с орнаментом ""волна"" снаружи...",rim
7,М102-2012-1-0752,Тарелка,тарелка,Керамика,ceramic,Фрагмент,fragment,"Сосуда (тарелки, блюда ?) красноглиняной с бел...",base
8,М102-2012-1-0827,Тарелка,тарелка,Фаянс,faience,Фрагмент,fragment,Сосуда (тарелки ?) фаянсового с белой поливой ...,rim
9,М102-2012-1-0833,Тарелка,тарелка,Фаянс,faience,Фрагмент,fragment,Сосуда (тарелки ?) фаянсового с белой поливой ...,rim


In [7]:
summary_tables = {
    'object_type': df['object_type'].value_counts(dropna=False),
    'material_group': df['material_group'].value_counts(dropna=False),
    'integrity': df['integrity'].value_counts(dropna=False),
    'part_zone': df['part_zone'].value_counts(dropna=False),
}

for name, series in summary_tables.items():
    print(f'\n{name}')
    print(series.to_string())


object_type
object_type
тарелка    484
изразец    361
блюдце     216
крышка     177
миска      117
плитка      26
игрушка      6

material_group
material_group
ceramic      630
faience      368
porcelain    307
glass         28
clay          26
other         14
stone          8
stoneware      3
metal          3

integrity
integrity
fragment    1258
whole        129

part_zone
part_zone
unknown    598
rim        265
profile    204
base       191
lid        112
wall        12
handle       5


In [8]:
normalization_policy = pd.DataFrame([
    {'case': 'missing value', 'encoding': 'unknown or dedicated missing flag'},
    {'case': 'rare category', 'encoding': 'map to broader stable class, usually other or parent material'},
    {'case': 'ambiguous label', 'encoding': 'keep coarse class and set label_is_uncertain=True where possible'},
    {'case': 'unrecognized part in description', 'encoding': 'part_zone=unknown'},
])
display(normalization_policy)

,case,encoding
0,missing value,unknown or dedicated missing flag
1,rare category,"map to broader stable class, usually other or ..."
2,ambiguous label,keep coarse class and set label_is_uncertain=T...
3,unrecognized part in description,part_zone=unknown


## Task 7. auto_description template and confidence rules

In [9]:
mandatory_fields = ['object_type', 'integrity']
optional_fields = ['part_zone', 'material_group']

material_text = {
    'ceramic': 'керамический',
    'clay': 'глиняный',
    'faience': 'фаянсовый',
    'porcelain': 'фарфоровый',
    'glass': 'стеклянный',
    'stoneware': 'из каменной массы',
    'stone': 'каменный',
    'metal': 'металлический',
    'other': None,
}

part_text = {
    'base': 'донце',
    'rim': 'венчик/край',
    'profile': 'профиль',
    'wall': 'стенка',
    'lid': 'крышка',
    'handle': 'ручка',
    'unknown': None,
}

integrity_text = {
    'fragment': 'фрагмент',
    'whole': 'целый предмет',
    'unknown': 'предмет',
}

def build_auto_description(fields, conf=None, thresholds=None):
    conf = conf or {}
    thresholds = thresholds or {'object_type': 0.0, 'integrity': 0.0, 'part_zone': 0.6, 'material_group': 0.65}

    obj = fields.get('object_type', 'предмет')
    integrity = fields.get('integrity', 'unknown')
    part_zone = fields.get('part_zone', 'unknown')
    material_group = fields.get('material_group', 'other')

    tokens = [obj, integrity_text.get(integrity, 'предмет')]

    if conf.get('part_zone', 1.0) >= thresholds['part_zone'] and part_text.get(part_zone):
        tokens.append(part_text[part_zone])

    if conf.get('material_group', 1.0) >= thresholds['material_group'] and material_text.get(material_group):
        tokens.append(material_text[material_group])

    return ', '.join(tokens)

print(build_auto_description({'object_type': 'тарелка', 'integrity': 'fragment', 'part_zone': 'rim', 'material_group': 'faience'}))

тарелка, фрагмент, венчик/край, фаянсовый


In [10]:
template_examples = [
    ({'object_type': 'изразец', 'integrity': 'fragment', 'part_zone': 'wall', 'material_group': 'ceramic'}, {'part_zone': 0.91, 'material_group': 0.88}),
    ({'object_type': 'тарелка', 'integrity': 'fragment', 'part_zone': 'rim', 'material_group': 'faience'}, {'part_zone': 0.90, 'material_group': 0.92}),
    ({'object_type': 'блюдце', 'integrity': 'fragment', 'part_zone': 'profile', 'material_group': 'porcelain'}, {'part_zone': 0.85, 'material_group': 0.83}),
    ({'object_type': 'крышка', 'integrity': 'whole', 'part_zone': 'lid', 'material_group': 'glass'}, {'part_zone': 0.87, 'material_group': 0.79}),
    ({'object_type': 'миска', 'integrity': 'fragment', 'part_zone': 'base', 'material_group': 'ceramic'}, {'part_zone': 0.71, 'material_group': 0.67}),
    ({'object_type': 'тарелка', 'integrity': 'fragment', 'part_zone': 'profile', 'material_group': 'faience'}, {'part_zone': 0.41, 'material_group': 0.58}),
]

rows = []
for fields, conf in template_examples:
    rows.append({
        'fields': fields,
        'conf': conf,
        'auto_description': build_auto_description(fields, conf=conf),
    })

display(pd.DataFrame(rows))

,fields,conf,auto_description
0,"{'object_type': 'изразец', 'integrity': 'fragm...","{'part_zone': 0.91, 'material_group': 0.88}","изразец, фрагмент, стенка, керамический"
1,"{'object_type': 'тарелка', 'integrity': 'fragm...","{'part_zone': 0.9, 'material_group': 0.92}","тарелка, фрагмент, венчик/край, фаянсовый"
2,"{'object_type': 'блюдце', 'integrity': 'fragme...","{'part_zone': 0.85, 'material_group': 0.83}","блюдце, фрагмент, профиль, фарфоровый"
3,"{'object_type': 'крышка', 'integrity': 'whole'...","{'part_zone': 0.87, 'material_group': 0.79}","крышка, целый предмет, крышка, стеклянный"
4,"{'object_type': 'миска', 'integrity': 'fragmen...","{'part_zone': 0.71, 'material_group': 0.67}","миска, фрагмент, донце, керамический"
5,"{'object_type': 'тарелка', 'integrity': 'fragm...","{'part_zone': 0.41, 'material_group': 0.58}","тарелка, фрагмент"


Правила включения полей в `auto_description`:

- `object_type` и `integrity` обязательны;
- `part_zone` включаем только при уверенности выше порога и если зона не `unknown`;
- `material_group` включаем только при уверенности выше порога и если это не `other`;
- при низкой уверенности модель лучше возвращает более грубое, но непротиворечивое описание без лишней детализации.

In [11]:
export_cols = [
    'code', 'image_file', 'object_type_raw', 'object_type', 'material_raw', 'material_group',
    'integrity_raw', 'integrity', 'part_zone_raw', 'part_zone',
    'label_is_missing_part_zone', 'label_is_missing_material',
    'label_is_uncertain_object_type', 'label_is_uncertain_part_zone',
]
out_path = REPORTS_DIR / 'baseline_targets_preview.csv'
df[export_cols].to_csv(out_path, index=False)
print(out_path)

/Users/artemvardanan/Downloads/DL_project1/artifacts/reports/baseline_targets_preview.csv
